# Hybrid Text Extraction — Colab GPU (Silver layer)

**Key optimisation:** ~60% of TBMM önerge PDFs carry a real text layer (no OCR needed). Only ~40% are true scans. This notebook tries native text extraction first (instant, 100% accurate, no GPU) and falls back to PaddleOCR only for scanned pages — halving GPU time and improving quality.

Per önerge:
1. Download PDF (parallel CDN)
2. `pypdf` text-layer extraction
3. If text length ≥ threshold → use it (`method='text-layer'`)
4. Else → PaddleOCR GPU (`method='ocr'`)
5. Write text row to Drive Silver shard
6. Delete PDF bytes (rolling ~150 MB disk)

**Prerequisites**
- `data/bronze/yazili_soru_meta.parquet` → Drive `/MyDrive/stat401-tbmm/bronze/`.
- `OCR_ONERGE_ONLY = True` (default) skips cevap PDFs.

**Outputs (Drive `/MyDrive/stat401-tbmm/silver/text/`)**
- `part-NNNN.parquet` — `(guid, doc_type, method, n_pages, text, n_chars, avg_conf)`
- `extract_done.txt` — resume key (`<guid>_<doc_type>`)
- `extract_errors.parquet`

## 1. Setup

In [ ]:
!nvidia-smi -L

In [ ]:
!apt-get install -y poppler-utils -qq
!pip install -q paddlepaddle-gpu "paddleocr<3.0" pdf2image pypdf pillow tqdm pyarrow pandas requests

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
import time, json, traceback, io, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from pypdf import PdfReader
from pdf2image import convert_from_bytes
from tqdm.auto import tqdm

# INPUT comes from public GitHub (no Drive needed). OUTPUT persists to Drive.
META_URL = 'https://raw.githubusercontent.com/esenbora/stat401-tbmm/main/data/yazili_soru_meta.parquet'
META_PATH = Path('/content/yazili_soru_meta.parquet')
PROJECT = Path('/content/drive/MyDrive/stat401-tbmm')   # output only
SILVER_DIR = PROJECT / 'silver' / 'text'
DONE_FILE = PROJECT / 'silver' / 'extract_done.txt'
ERR_FILE = PROJECT / 'silver' / 'extract_errors.parquet'

OCR_ONERGE_ONLY = True       # skip cevap PDFs
TEXT_LAYER_MIN_CHARS = 100   # below this -> treat as scanned, OCR
BATCH_SIZE = 500
DPI = 250
MAX_PAGES = 8
DL_WORKERS = 8
DL_TIMEOUT = 60

SILVER_DIR.mkdir(parents=True, exist_ok=True)
DONE_FILE.parent.mkdir(parents=True, exist_ok=True)
H = {'User-Agent': 'Mozilla/5.0', 'Accept-Language': 'tr-TR,tr;q=0.9'}

# fetch metadata from GitHub raw
if not META_PATH.exists():
    print('downloading metadata from GitHub...')
    urllib.request.urlretrieve(META_URL, META_PATH)
print('META exists:', META_PATH.exists(), META_PATH.stat().st_size//1024, 'KB')

## 2. Build job list + resume

In [ ]:
meta = pd.read_parquet(META_PATH)
print(f'metadata rows: {len(meta)}')

jobs = []
for _, r in meta.iterrows():
    if r.get('onerge_pdf_url'):
        jobs.append({'guid': r['guid'], 'doc_type': 'onerge', 'url': r['onerge_pdf_url']})
    if not OCR_ONERGE_ONLY and r.get('cevap_pdf_url'):
        jobs.append({'guid': r['guid'], 'doc_type': 'cevap', 'url': r['cevap_pdf_url']})
print(f'total jobs: {len(jobs)}')

done = set(DONE_FILE.read_text().splitlines()) if DONE_FILE.exists() else set()
todo = [j for j in jobs if f"{j['guid']}_{j['doc_type']}" not in done]
print(f'resume — done: {len(done)}, todo: {len(todo)}')

## 3. PaddleOCR init

In [ ]:
from paddleocr import PaddleOCR
ocr = PaddleOCR(use_angle_cls=True, lang='latin', use_gpu=True, show_log=False)
print('PaddleOCR ready')

## 4. Extraction workers (text-layer first, OCR fallback)

In [ ]:
def download(job):
    try:
        r = requests.get(job['url'], headers=H, timeout=DL_TIMEOUT)
        r.raise_for_status()
        return job, r.content, ''
    except Exception as e:
        return job, None, str(e)[:200]

def try_text_layer(pdf_bytes):
    """Return (text, n_pages) if a usable text layer exists, else (None, n_pages)."""
    rd = PdfReader(io.BytesIO(pdf_bytes))
    n = len(rd.pages)
    parts = []
    for pg in rd.pages[:MAX_PAGES]:
        parts.append(pg.extract_text() or '')
    txt = '\n'.join(parts).strip()
    if len(txt) >= TEXT_LAYER_MIN_CHARS:
        return txt, n
    return None, n

def ocr_bytes(pdf_bytes):
    imgs = convert_from_bytes(pdf_bytes, dpi=DPI)[:MAX_PAGES]
    texts, confs = [], []
    for img in imgs:
        res = ocr.ocr(np.array(img), cls=True)
        lines = res[0] if res else []
        texts.append('\n'.join(ln[1][0] for ln in lines))
        confs.extend(float(ln[1][1]) for ln in lines)
    return '\n'.join(texts), len(imgs), (float(np.mean(confs)) if confs else 0.0)

def process(job, pdf_bytes):
    """Returns a single result row dict."""
    text, n_pages = try_text_layer(pdf_bytes)
    if text is not None:
        return {
            'guid': job['guid'], 'doc_type': job['doc_type'], 'method': 'text-layer',
            'n_pages': n_pages, 'text': text, 'n_chars': len(text), 'avg_conf': 1.0,
        }
    # scanned → OCR
    text, n_pages, conf = ocr_bytes(pdf_bytes)
    return {
        'guid': job['guid'], 'doc_type': job['doc_type'], 'method': 'ocr',
        'n_pages': n_pages, 'text': text, 'n_chars': len(text), 'avg_conf': conf,
    }

## 5. Main loop — parallel download, sequential extract, checkpoint

In [ ]:
existing = sorted(SILVER_DIR.glob('part-*.parquet'))
next_idx = int(existing[-1].stem.split('-')[1]) + 1 if existing else 0
print(f'next shard: {next_idx:04d}')

def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def flush(rows, idx):
    if not rows: return
    pd.DataFrame(rows).to_parquet(SILVER_DIR / f'part-{idx:04d}.parquet', index=False)

def append_done(keys):
    with DONE_FILE.open('a') as f:
        for k in keys: f.write(k + '\n')

def append_err(errs):
    if not errs: return
    new = pd.DataFrame(errs)
    if ERR_FILE.exists():
        new = pd.concat([pd.read_parquet(ERR_FILE), new], ignore_index=True)
    new.to_parquet(ERR_FILE, index=False)

stats = {'text-layer': 0, 'ocr': 0}
with tqdm(total=len(todo), desc='hybrid extract') as pbar:
    for batch in chunks(todo, BATCH_SIZE):
        downloaded = []
        errors = []
        with ThreadPoolExecutor(max_workers=DL_WORKERS) as ex:
            for fut in as_completed([ex.submit(download, j) for j in batch]):
                j, content, err = fut.result()
                if content is None:
                    errors.append({'key': f"{j['guid']}_{j['doc_type']}", 'stage': 'download', 'error': err})
                else:
                    downloaded.append((j, content))
        rows, keys = [], []
        for j, content in downloaded:
            key = f"{j['guid']}_{j['doc_type']}"
            try:
                row = process(j, content)
                stats[row['method']] += 1
                rows.append(row)
                keys.append(key)
            except Exception as e:
                errors.append({'key': key, 'stage': 'extract', 'error': repr(e)[:200]})
            pbar.update(1)
        flush(rows, next_idx)
        append_done(keys)
        append_err(errors)
        next_idx += 1
        pbar.set_postfix(text=stats['text-layer'], ocr=stats['ocr'])

print('done', stats)

## 6. Sanity check

In [ ]:
shards = sorted(SILVER_DIR.glob('part-*.parquet'))
if shards:
    df = pd.concat([pd.read_parquet(s) for s in shards], ignore_index=True)
    print(f'shards: {len(shards)}  rows: {len(df)}')
    print('method split:', df.method.value_counts().to_dict())
    print(f'avg chars/doc: {df.n_chars.mean():.0f}')
    print(f'OCR avg conf: {df[df.method=="ocr"].avg_conf.mean():.3f}')
    df.to_parquet(PROJECT / 'silver' / 'onerge_text.parquet', index=False)
    print('consolidated → silver/onerge_text.parquet')